In [1]:
:set -XOverloadedStrings
:set -XScopedTypeVariables
:set -XFlexibleContexts
:load ../lib/src/Quantale.hs ../lib/src/KanExtension.hs ../lib/src/Bitopos.hs ../lib/src/Distribution.hs ../lib/src/SubjectiveModel.hs
import Quantale
import KanExtension
import Bitopos
import Distribution
import SubjectiveModel
import Data.List (sortBy, maximumBy, minimumBy, intercalate)
import Data.Ord (comparing, Down(..))
import Data.Maybe (fromMaybe)
import Text.Printf (printf)
import System.IO (hSetBuffering, stdout, BufferMode(..))
hSetBuffering stdout LineBuffering
putStrLn "\x2705 SETUP OK: applied subjective modelling (Godel / Goguen / Luka scales from lib)"

✅ SETUP OK: applied subjective modelling (Godel / Goguen / Luka scales from lib)

# 🎚️ Субъективное Моделирование: Приложения

> **Апогей триптиха.** [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) ввёл меры правдоподобия $\mathrm{Pl}$ и доверия $\mathrm{Bel}$; [PytevIso.ipynb](PytevIso.ipynb) доказал, что все категорные представления этих мер изоморфны, и в §9 показал, что **вся теория параметризована кванталью** $L$. Здесь — расплата за страдания: вся тяжёлая математика превращается в простой инструмент. Мы берём одно достаточно богатое дело и проводим его через **три мира** — три варианта теории Пытьева (§1.9 его пособия-2022) — и смотрим, где они соглашаются, а где спорят.

**Девиз — Эрлангенская программа для неопределённости.** Феликс Клейн определил геометрию как теорию инвариантов группы преобразований: евклидова геометрия — инварианты движений, аффинная — инварианты аффинной группы, топология — инварианты гомеоморфизмов. **Точно так же вариант теории свидетельств = геометрия группы автоморфизмов шкалы $\mathrm{Aut}(L)$.** Чем меньше группа, тем больше содержательных инвариантов, тем «жёстче» мир. Наш сравнительный анализ — это буквально «что инвариантно во всех трёх геометриях» (мета-конструкция) против «что зависит от выбора мира» (артефакт шкалы).

**Практический итог:** одно и то же дело, один и тот же код (библиотека полиморфна по квантали) — три вердикта. Их совпадения — то, чему можно доверять безусловно; их расхождения показывают, какое свойство дела задело именно *вариантную* часть теории. Так мы отделяем мух от котлет — и тех, кто мне верит, от тех, кто нет.

## 📌 Содержание

| № | Раздел | Мир / статус |
|---|--------|--------------|
| 1 | Пролог: карта миров и Эрлангенская программа | обзор |
| 2 | Дело о пяти подозреваемых (и контрольный пример — двигатель) | постановка |
| 3 | Мир Гёделя $(\max, \min)$: слабейшее звено | ✅ вариант 1 |
| 4 | Мир Гогена $(\max, \cdot)$: накопление улик и лог-инварианты | ⏳ вариант 3 |
| 5 | Зарубки: коллегия договаривается о порогах | ⏳ вариант 2 |
| 6 | Мир Лукасевича (MV): вишенка — мост к вероятности | ⏳ вариант 4 |
| 7 | Сравнительный анализ: мухи и котлеты | ⏳ свод |

> **Как читать.** Разделы 3–6 — один и тот же код на разных шкалах (библиотека полиморфна). Смотрите не на абсолютные числа (они содержательны лишь до автоморфизма шкалы!), а на **порядок подозреваемых** и на то, **где он меняется между мирами**.

# 1. Пролог: Карта Миров

Один математический аппарат — sup-меры $\mathrm{Pl}(E) = \sup_{x\in E}\tau(x)$ и дуальные $\mathrm{Bel}$ — но **алгебра шкалы** $L$ бывает разной. Пытьев в §1.9 пособия-2022 описывает три варианта; в [PytevIso.ipynb](PytevIso.ipynb), §9 доказано, что это три подстановки квантали в одну мета-конструкцию. Библиотека курса реализует их как инстансы класса `Quantale` — и весь код ниже полиморфен.

| Мир | Кванталь $L$ | $\otimes$ (комбинирование улик) | $a \multimap b$ (кондиционирование) | $\mathrm{Aut}(L)$ | Характер |
|---|---|---|---|---|---|
| **Гёдель** (вар. 1) | $([0,1], \max, \min)$ | $\min$ — слабейшее звено | усечение | все строго монотонные биекции | осторожный, шумоустойчивый, «забывает» серию слабых улик |
| **Гоген** (вар. 3) | $([0,1], \max, \cdot)$ | $\cdot$ — накопление | деление | степенные $a \mapsto a^\alpha$ | психофизический; серия слабых улик топит; есть лог-инварианты |
| **Зарубки** (вар. 2) | Гёдель $+$ константы $S$ | $\min$ | усечение | фиксирующие $S$ | коллективный: договорённые пороги содержательны для всех |
| **Лукасевич** (вар. 4) | $([0,1], \max, \max(0,a{+}b{-}1))$ | граница Фреше | $\min(1,1{-}a{+}b)$ | только $\mathrm{id}$ | абсолютная шкала; ближе всех к вероятности (мост в §6) |

**Лестница инвариантов (Эрланген).** Группы вложены: $\mathrm{Aut}(\text{Гёдель}) \supset \Gamma_S \supset \mathrm{Aut}(\text{Гоген}) \supset \mathrm{Aut}(\text{Лукасевич}) = \{\mathrm{id}\}$. Соответственно растёт список содержательного:

- Гёдель: осмысленен **только порядок** подозреваемых ($x$ подозрительнее $y$);
- зарубки: порядок **+ положение относительно порогов** («выше разумного сомнения»);
- Гоген: **+ лог-отношения** $\log\mathrm{Pl}(A)/\log\mathrm{Pl}(B)$ («во сколько раз подозрительнее»);
- Лукасевич: **все значения** абсолютны — это и есть вероятностный предел.

$0$ (невозможно) и $1$ (достоверно) инвариантны в любом мире — единственные абсолюты, общие всем (см. врезку о нулевых множествах в [PytevIso.ipynb](PytevIso.ipynb), §8). Отсюда и стратегия анализа: **вывод, устойчивый во всех мирах, опирается только на порядок и нули — ему можно верить; вывод, что рушится при смене мира, держался на артефакте шкалы.**

# 2. Дело о Пяти Подозреваемых

Детектив-модельер расследует кражу. Круг сужен до пяти человек:

| Код | Подозреваемый | Базовое правдоподобие $\tau$ | Комментарий детектива |
|---|---|---|---|
| A | Alice | $0.9$ | мотив и возможность, видели рядом |
| B | Bob | $0.6$ | есть мотив, но алиби частичное |
| C | Carol | $0.6$ | подозрительное поведение |
| D | Dave | $0.3$ | слабый мотив |
| E | Erin | $0.0$ | **железное алиби — была в другом городе** |

Число $\tau(x)$ — субъективная оценка детектива, «насколько правдоподобна виновность $x$» в *ранговой* шкале: содержателен порядок, а не абсолютная величина (Гёдель). Erin с $\tau = 0$ образует **нулевую зону** $Z$: событие «виновна Erin» невозможно, и носитель дела — $X \setminus Z = \{A,B,C,D\}$ (см. врезку о нулевых множествах, [PytevIso.ipynb](PytevIso.ipynb), §8).

**Дуальная согласованность.** Доверие берём согласованным: $\bar\tau = \theta\circ\tau$ (в Гёделе $\theta(t) = 1-t$; в Лукасевиче эта инволюция окажется *внутренней* — §6). Тогда $\mathrm{Bel}(E) = \inf_{x\notin E}\bar\tau(x)$ — доверие к тому, что виновный внутри $E$.

**Три улики** приходят по ходу следствия; каждая задаёт *совместимость* $c_i(x) \in [0,1]$ подозреваемого $x$ с уликой:

| Улика | A | B | C | D | E |
|---|---|---|---|---|---|
| $c_1$: след ботинка 44-го размера | $0.9$ | $0.7$ | $0.5$ | $0.7$ | $0.4$ |
| $c_2$: синее волокно на месте | $0.5$ | $0.7$ | $0.8$ | $0.6$ | $0.3$ |
| $c_3$: замечен у дома в 21:00 | $0.6$ | $0.7$ | $0.4$ | $0.9$ | $0.2$ |

Комбинирование улики со свидетельством — тензор шкалы: $\tau'(x) = \tau(x) \otimes c_1(x) \otimes c_2(x) \otimes c_3(x)$. **Именно здесь миры расходятся сильнее всего:** в Гёделе $\otimes = \min$ (итог — слабейшее звено, серия улик не накапливается), в Гогене $\otimes = \cdot$ (улики перемножаются, серия слабых топит). Обратите внимание на Bob: три *средних* улики ($0.7$ каждая) — в Гёделе он останется на $0.6$, а в Гогене серия его утопит. Кто «подозрительнее» — зависит от мира; какой именно вопрос это выявит — увидим в §7.

**Контрольный пример (строгий) — диагностика двигателя.** Параллельно ведём технический пример из [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb): НОЭ $\tilde x$ — неисправный узел, $\tau$ — правдоподобие по показаниям датчиков. Он даёт «сухую» проверку тех же формул там, где детективная фабула могла бы усыпить бдительность. Его числа появятся в сравнительной таблице §7.

# 3. Мир Гёделя: Слабейшее Звено

Первый вариант Пытьева — шкала $L = ([0,1], \max, \min)$. Это исходная теория возможностей: макситивные $\mathrm{Pl}$, минитивные $\mathrm{Bel}$, комбинирование улик через $\min$. Работаем полиморфной моделью `SubjModelQ UnitInterval` из библиотеки (`UnitInterval` — это и есть шкала Гёделя).

**Что вычисляем — весь инструментарий предыдущих ноутбуков в деле:**

- $\mathrm{Pl}(E) = \sup_{x\in E}\tau(x)$ и $\mathrm{Bel}(E) = \inf_{x\notin E}\bar\tau(x)$ — `smqPl`/`smqBel` (расширения Кана из ядра, [PytevIso.ipynb](PytevIso.ipynb), §1);
- **носитель** $X\setminus Z$ — главный фильтр достоверности (§8 PytevIso): Erin отсеивается;
- **решение** — подозреваемый максимального правдоподобия и «зазор разумного сомнения» $\mathrm{Pl}(\text{топ}) $ против $\mathrm{Pl}(\text{остальные})$;
- **комбинирование улик** через тензор $\otimes = \min$: `qTensor` на `UnitInterval`.

**Характер мира.** $\min$ — «слабейшее звено»: итоговое правдоподобие подозреваемого не ниже, чем его худшая совместимость с уликами, но и серию средних улик $\min$ **не накапливает** — три улики по $0.7$ дают ровно $0.7$. Отсюда осторожность и шумоустойчивость: одна ошибочная сильная улика не раздувает подозрение, но и «гора мелких косвенных» не давит. Ключевая деталь ниже — **тот же код** (`combine`) в §4 получит `goguen` вместо `ui`, и мир сменится без единой правки логики.

In [2]:
-- | Раздел 3: дело в мире Гёделя. Данные дела и полиморфная машинерия
--   определяются здесь ОДИН раз — разделы 4/6 переиспружают их, меняя лишь
--   инъекцию шкалы (ui -> goguen -> luka): один код, три мира.

suspects :: [Char]
suspects = "ABCDE"

nameOf :: Char -> String
nameOf c = fromMaybe [c] (lookup c
  [('A', "Alice"), ('B', "Bob"), ('C', "Carol"), ('D', "Dave"), ('E', "Erin")])

val :: [(Char, Double)] -> Char -> Double
val tab c = fromMaybe 0 (lookup c tab)

-- Базовое правдоподобие и три улики (совместимости) — общие для всех миров.
baseTab :: [(Char, Double)]
baseTab = [('A', 0.9), ('B', 0.6), ('C', 0.6), ('D', 0.3), ('E', 0.0)]

clues :: [(String, [(Char, Double)])]
clues =
  [ ("sled botinka 44", [('A', 0.9), ('B', 0.7), ('C', 0.5), ('D', 0.7), ('E', 0.4)])
  , ("sinee volokno",   [('A', 0.5), ('B', 0.7), ('C', 0.8), ('D', 0.6), ('E', 0.3)])
  , ("u doma v 21:00",  [('A', 0.6), ('B', 0.7), ('C', 0.4), ('D', 0.9), ('E', 0.2)]) ]

-- Полиморфная свёртка улик в квантали: tau(x) (x) c1(x) (x) c2(x) (x) c3(x).
combineWith :: Quantale q => (Double -> q) -> Char -> q
combineWith inj c = foldr qTensor (inj (val baseTab c)) [ inj (val tab c) | (_, tab) <- clues ]

-- Модель дела в инволютивной квантали (нужна для Bel-стороны).
caseModel :: InvolutiveQuantale q => (Double -> q) -> SubjModelQ q Char
caseModel inj = dualConsistentQ suspects (inj . val baseTab)

-- Ранжирование по убыванию: ПОРЯДОК — то, что инвариантно между мирами.
ranking :: Ord q => (Char -> q) -> [(Char, q)]
ranking f = sortBy (comparing (Down . snd)) [ (c, f c) | c <- suspects ]

-- --- Гёдель: шкала UnitInterval (max, min) ---
modelG :: SubjModelQ UnitInterval Char
modelG = caseModel ui

plG, belG :: [Char] -> Double
plG  e = unUI (smqPl modelG e)
belG e = unUI (smqBel modelG e)

demoGodel :: IO ()
demoGodel = do
  putStrLn "=== Mir Godelya (max, min): tenzor ulik = min ==="
  putStrLn "-- Bazovoe pravdopodobie (soderzhatelen poryadok, ne chisla) --"
  mapM_ (\(c, v) -> printf "  %-6s tau = %.2f\n" (nameOf c) (unUI v))
        (ranking (ui . val baseTab))
  printf "  Nositel (X \\ Z, nulevaya zona Erin otsechena): %s\n"
         (intercalate ", " [ nameOf c | c <- suspects, val baseTab c > 0 ])
  putStrLn "-- Mery sobytij  Pl / Bel --"
  let evs = [ ("{Alice}", "A"), ("{Alice,Bob}", "AB")
            , ("kulprit ne Erin (ABCD)", "ABCD"), ("{Erin}", "E") ]
  mapM_ (\(nm, e) -> printf "  %-24s Pl = %.2f   Bel = %.2f\n" nm (plG e) (belG e)) evs
  putStrLn "-- Posle tryoh ulik:  tau (x) c1 (x) c2 (x) c3  (v Godele = min) --"
  mapM_ (\(c, v) -> printf "  %-6s -> %.2f\n" (nameOf c) (unUI v)) (ranking (combineWith ui))
  putStrLn "  (!) Bob: tri sredniye uliki 0.7 NE nakaplivayutsya (min = 0.6);"
  putStrLn "      Alice tonet do 0.5 iz-za slabogo zvena (sinee volokno 0.5)."
  let scored = [ (c, unUI (combineWith ui c)) | c <- suspects ]
      (best, bv) = maximumBy (comparing snd) scored
      rest = maximum [ v | (c, v) <- scored, c /= best ]
  printf "  Reshenie Godelya: %s (Pl=%.2f), zazor razumnogo somneniya: %.2f\n"
         (nameOf best) bv (bv - rest)

demoGodel

=== Mir Godelya (max, min): tenzor ulik = min ===
-- Bazovoe pravdopodobie (soderzhatelen poryadok, ne chisla) --
  Alice  tau = 0.90
  Bob    tau = 0.60
  Carol  tau = 0.60
  Dave   tau = 0.30
  Erin   tau = 0.00
  Nositel (X \ Z, nulevaya zona Erin otsechena): Alice, Bob, Carol, Dave
-- Mery sobytij  Pl / Bel --
  {Alice}                  Pl = 0.90   Bel = 0.40
  {Alice,Bob}              Pl = 0.90   Bel = 0.40
  kulprit ne Erin (ABCD)   Pl = 0.90   Bel = 1.00
  {Erin}                   Pl = 0.00   Bel = 0.10
-- Posle tryoh ulik:  tau (x) c1 (x) c2 (x) c3  (v Godele = min) --
  Bob    -> 0.60
  Alice  -> 0.50
  Carol  -> 0.40
  Dave   -> 0.30
  Erin   -> 0.00
  (!) Bob: tri sredniye uliki 0.7 NE nakaplivayutsya (min = 0.6);
      Alice tonet do 0.5 iz-za slabogo zvena (sinee volokno 0.5).
  Reshenie Godelya: Bob (Pl=0.60), zazor razumnogo somneniya: 0.10

---

**Прикладная сноска** к триптиху [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) → [PytevIso.ipynb](PytevIso.ipynb) → (этот ноутбук). Библиотека: [darklordshish/SubjectiveModeling](https://github.com/darklordshish/SubjectiveModeling). ↩ [Путеводитель](../README.ipynb)